In [2]:
import pandas as pd
import numpy as np
import warnings
import gc 
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
y_test = pd.read_csv('sample_submission.csv')

In [4]:
y_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172585 entries, 0 to 172584
Data columns (total 2 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             172585 non-null  int64  
 1   accident_risk  172585 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 2.6 MB


In [5]:
train.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [6]:
test.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172585 entries, 0 to 172584
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      172585 non-null  int64  
 1   road_type               172585 non-null  object 
 2   num_lanes               172585 non-null  int64  
 3   curvature               172585 non-null  float64
 4   speed_limit             172585 non-null  int64  
 5   lighting                172585 non-null  object 
 6   weather                 172585 non-null  object 
 7   road_signs_present      172585 non-null  bool   
 8   public_road             172585 non-null  bool   
 9   time_of_day             172585 non-null  object 
 10  holiday                 172585 non-null  bool   
 11  school_season           172585 non-null  bool   
 12  num_reported_accidents  172585 non-null  int64  
dtypes: bool(4), float64(1), int64(4), object(4)
memory usage: 12.5+ MB


In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

In [9]:
print(f"Размер обучающей выборки: {train.shape}")
print(f"Размер тестовой выборки: {test.shape}")

Размер обучающей выборки: (517754, 14)
Размер тестовой выборки: (172585, 13)


In [10]:
train.duplicated().sum()

0

In [11]:
# Создаем функцию для единообразной обработки трейна и теста
def preprocess_data(df):
    data = df.copy()
    
    # Генерация новых фичей
    data['accidents_per_lane'] = data['num_reported_accidents'] / (data['num_lanes'] + 1e-5)
    data['speed_curvature_interaction'] = data['speed_limit'] * data['curvature']
    data['lane_speed_ratio'] = data['num_lanes'] / (data['speed_limit'] + 1e-5)
    data['log_reported_accidents'] = np.log1p(data['num_reported_accidents'])
    
    # ВНИМАНИЕ: Здесь теперь переводим в целые числа (int), а не в category
    # True станет 1, False станет 0
    data['is_high_speed'] = (data['speed_limit'] > 60).astype(int) 
    
    return data

print("Генерация новых признаков...")
train_features = preprocess_data(train)
test_features = preprocess_data(test)

print(f"Количество признаков: {train_features.shape[1]}")

Генерация новых признаков...
Количество признаков: 19


In [12]:
# Разделяем признаки на текстовые и булевые
text_features = ['road_type', 'lighting', 'weather', 'time_of_day']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']

# 1. Текстовые (строковые) переводим в category
for col in text_features:
    train_features[col] = train_features[col].astype('category')
    test_features[col] = test_features[col].astype('category')

# 2. Изначально булевые (True/False) признаки просто переводим в int (1/0)
for col in bool_features:
    train_features[col] = train_features[col].astype(int)
    test_features[col] = test_features[col].astype(int)

# Выделяем целевую переменную и ID
target_col = 'accident_risk'

# Окончательно формируем матрицы для обучения
features = [col for col in train_features.columns if col not in ['id', target_col]]

X = train_features[features]
y = train_features[target_col]
X_test = test_features[features]

# Очищаем память от дубликатов
import gc
del train, test, train_features, test_features
gc.collect()

print("Готово! Данные обработаны и безопасны для XGBoost.")

Готово! Данные обработаны и безопасны для XGBoost.


In [54]:
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import xgboost as xgb

def objective(trial):
    # Указываем, какие параметры и в каких границах будет перебирать алгоритм
    param = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        # Количество деревьев берем с запасом, контролировать переобучение будет early_stopping
        'n_estimators': 3000, 
        
        # Интеллектуальный перебор гиперпараметров
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Делаем валидацию на 3 фолдах для ускорения поиска (потом обучим финальную на 5)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**param)
        
        # Обучаем (optuna callback может останавливать плохие попытки заранее)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            # Ранняя остановка: если 50 деревьев подряд качество не растет - прекращаем
        )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores) # Возвращаем среднюю ошибку - Optuna попытается сделать ее минимальной

In [ ]:
print("Запускаем умный подбор гиперпараметров...")
# Создаем процесс оптимизации (direction='minimize' потому что хотим уменьшить RMSE)
study = optuna.create_study(direction='minimize', study_name="XGB_tuning")

# Запускаем перебор. n_trials = количество попыток. 
# 30-50 достаточно для хорошего результата. Поставишь 100 и оставишь на ночь — будет топ.
study.optimize(objective, n_trials=30) 

print("Лучшие параметры найдены!")
print(f"Лучший Score (RMSE): {study.best_value}")
print("Лучшие параметры:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

[I 2026-05-11 17:34:03,823] A new study created in memory with name: XGB_tuning


Запускаем умный подбор гиперпараметров...


[I 2026-05-11 17:40:10,437] Trial 0 finished with value: 0.05657209086471696 and parameters: {'learning_rate': 0.001069587051035914, 'max_depth': 8, 'subsample': 0.6707044416747769, 'colsample_bytree': 0.9780038766757204, 'min_child_weight': 3, 'reg_alpha': 9.319840357802524e-08, 'reg_lambda': 6.583751144404721e-08}. Best is trial 0 with value: 0.05657209086471696.
[I 2026-05-11 17:45:59,570] Trial 1 finished with value: 0.060843652138976 and parameters: {'learning_rate': 0.05521065605446419, 'max_depth': 10, 'subsample': 0.6681047883592266, 'colsample_bytree': 0.5570783727887293, 'min_child_weight': 6, 'reg_alpha': 3.082724956216849e-07, 'reg_lambda': 1.403033586411545e-07}. Best is trial 0 with value: 0.05657209086471696.


In [ ]:
# Забираем лучшие параметры от Optuna
best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'hist'
best_params['enable_categorical'] = True
best_params['n_estimators'] = 5000 # Ставим еще больше деревьев для финала
best_params['random_state'] = 42
best_params['n_jobs'] = -1

# Возвращаемся к надежной кросс-валидации (5 фолдов)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X)) 
test_preds = np.zeros(len(X_test)) 

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Финальное обучение: Фолд {fold + 1}/5 ---")
    
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**best_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=500 # Выводить инфо раз в 500 деревьев
    )
    
    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    
    print(f"RMSE фолда {fold+1}: {np.sqrt(mean_squared_error(y_val, val_pred)):.5f}\n")
    
    # Усредняем предсказания на тестовой выборке
    test_preds += model.predict(X_test) / kf.n_splits

print(f"Итоговый OOF RMSE: {np.sqrt(mean_squared_error(y, oof_preds)):.5f}")

In [ ]:
# Создаем итоговый DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'accident_risk': test_preds
})

# Жестко ограничиваем вероятности границами от 0 до 1 (защита от "дурака", так как деревья могут дать выброс)
submission['accident_risk'] = np.clip(submission['accident_risk'], 0, 1)

# Сохраняем и скачиваем!
submission.to_csv('optuna_xgboost_submission.csv', index=False)

print("Твой самый сильный сабмит (optuna_xgboost_submission.csv) сохранен!")
submission.head()

In [14]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("Запуск быстрого обучения с золотыми настройками...")

# "Золотые" гиперпараметры. ВАЖНО: early_stopping_rounds теперь здесь!
fast_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',              
    'enable_categorical': True, 
    'n_estimators': 2000,               
    'learning_rate': 0.05,              
    'max_depth': 6,                     
    'subsample': 0.8,                   
    'colsample_bytree': 0.8,            
    'min_child_weight': 3,
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50         # <--- Перенесли сюда!
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X)) 
test_preds = np.zeros(len(X_test)) 

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Обучение Фолд {fold + 1}/5 ---")
    
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Модель создается уже со знанием того, что надо рано останавливаться
    model = xgb.XGBRegressor(**fast_params)
    
    # Обучаем!
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100  # Печатает лог каждые 100 деревьев
    )
    
    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    
    # В новых версиях лучшее количество деревьев лежит тут
    best_iter = model.best_iteration
    print(f"Фолд {fold+1} готов! Лучшее кол-во деревьев: {best_iter}. RMSE: {np.sqrt(mean_squared_error(y_val, val_pred)):.5f}\n")
    
    test_preds += model.predict(X_test) / kf.n_splits

print(f"Итоговое качество на кросс-валидации (OOF RMSE): {np.sqrt(mean_squared_error(y, oof_preds)):.5f}")

# --- Формируем файл сабмита ---
submission = pd.DataFrame({
    'id': pd.read_csv('test.csv')['id'], 
    'accident_risk': test_preds
})

submission['accident_risk'] = np.clip(submission['accident_risk'], 0, 1)

submission.to_csv('fast_xgboost_submission.csv', index=False)
print("✅ Готово! Файл 'fast_xgboost_submission.csv' сохранен и готов к отправке.")
submission.head()

Запуск быстрого обучения с золотыми настройками...
--- Обучение Фолд 1/5 ---
[0]	validation_0-rmse:0.15910
[100]	validation_0-rmse:0.05651
[200]	validation_0-rmse:0.05631
[300]	validation_0-rmse:0.05626
[400]	validation_0-rmse:0.05624
[500]	validation_0-rmse:0.05624
[537]	validation_0-rmse:0.05624
Фолд 1 готов! Лучшее кол-во деревьев: 487. RMSE: 0.05624

--- Обучение Фолд 2/5 ---
[0]	validation_0-rmse:0.15935
[100]	validation_0-rmse:0.05638
[200]	validation_0-rmse:0.05620
[300]	validation_0-rmse:0.05615
[400]	validation_0-rmse:0.05614
[442]	validation_0-rmse:0.05614
Фолд 2 готов! Лучшее кол-во деревьев: 392. RMSE: 0.05614

--- Обучение Фолд 3/5 ---
[0]	validation_0-rmse:0.15976
[100]	validation_0-rmse:0.05640
[200]	validation_0-rmse:0.05620
[300]	validation_0-rmse:0.05616
[400]	validation_0-rmse:0.05615
[440]	validation_0-rmse:0.05615
Фолд 3 готов! Лучшее кол-во деревьев: 390. RMSE: 0.05615

--- Обучение Фолд 4/5 ---
[0]	validation_0-rmse:0.15892
[100]	validation_0-rmse:0.05625
[200]	v

,id,accident_risk
0,517754,0.294046
1,517755,0.122653
2,517756,0.181034
3,517757,0.317652
4,517758,0.395077
